# 한이음 NLU 공정 비교 실험
## Hierarchical Gated Multi-Task koELECTRA + R-Drop

- 입력: `text`
- 출력: `intent`, `menu`, `temperature`, `quantity`
- 구조: **하나의 koELECTRA encoder + task별 attention pooling + intent-conditioned slot heads**
- 학습: **R-Drop consistency regularization**과 ORDER 전용 slot masking
- 데이터 재분할·정규화·중복 제거 없음
- best epoch: validation `frame_accuracy` 우선, 동률이면 `intent_macro_f1`
- test는 best model 선택이 끝난 뒤 정확히 한 번 최종 평가

> 이 구조는 기존의 task별 독립 encoder 4개 방식과 달리 하나의 encoder에서 표현을 공유하면서도, 각 task가 서로 다른 토큰에 주목하도록 설계했습니다.

## 1. 호환 패키지 설치

Google Colab T4 기준입니다. 아래 셀 실행 전에 `torch`, `transformers`, `sklearn`을 이미 import했다면 **런타임 → 세션 다시 시작** 후 이 셀부터 다시 실행하세요. 아직 import하지 않았다면 재시작 없이 계속 진행해도 됩니다.

In [ ]:
%pip -q install -U \
  "transformers==4.46.3" \
  "accelerate==1.1.1" \
  "scikit-learn==1.6.1" \
  "sentencepiece==0.2.0"

## 2. train.csv, valid.csv, test.csv 업로드

In [ ]:
from google.colab import files

print("train.csv, valid.csv, test.csv 세 파일을 선택하세요.")
uploaded = files.upload()

required_files = {"train.csv", "valid.csv", "test.csv"}
missing_files = required_files - set(uploaded.keys())
if missing_files:
    raise FileNotFoundError(f"업로드되지 않은 파일: {sorted(missing_files)}")
print("업로드 완료:", sorted(required_files))

## 3. CSV 불러오기 — 수정·정규화·재분할 없음

In [ ]:
import pandas as pd

# 모든 열을 문자열로 읽어 quantity 숫자/문자 혼합 문제를 방지한다.
# keep_default_na=False로 NONE 또는 빈 문자열을 임의의 NaN으로 바꾸지 않는다.
train_df = pd.read_csv("train.csv", dtype=str, keep_default_na=False)
valid_df = pd.read_csv("valid.csv", dtype=str, keep_default_na=False)
test_df  = pd.read_csv("test.csv",  dtype=str, keep_default_na=False)

print("train:", train_df.shape)
print("valid:", valid_df.shape)
print("test :", test_df.shape)
display(train_df.head())

## 4. 필수 컬럼 존재 여부만 검사

In [ ]:
REQUIRED_COLUMNS = ["text", "label", "menu", "temperature", "quantity"]

for split_name, df in {
    "train": train_df,
    "valid": valid_df,
    "test": test_df,
}.items():
    missing = [c for c in REQUIRED_COLUMNS if c not in df.columns]
    if missing:
        raise ValueError(f"{split_name}.csv 필수 컬럼 누락: {missing}")

print("필수 컬럼 검사 통과")

## 5. import, 실험 설정, random seed 고정

In [ ]:
import os
import gc
import json
import math
import random
import shutil
import zipfile
from pathlib import Path
from dataclasses import dataclass

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import transformers
import accelerate

from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_recall_fscore_support,
)
from tqdm.auto import tqdm

SEED = 42
EXPERIMENT_NAME = "hierarchical_gated_multitask_koelectra_rdrop"
MODEL_ARCHITECTURE = (
    "Shared koELECTRA encoder + task-specific gated attention pooling + "
    "intent-conditioned slot heads + R-Drop"
)
ENCODER_NAME = "monologg/koelectra-small-v3-discriminator"

EPOCHS = 10
BATCH_SIZE = 16
LEARNING_RATE = 3e-5
WEIGHT_DECAY = 0.01
MAX_LENGTH = 64
WARMUP_RATIO = 0.1
GRAD_CLIP_NORM = 1.0
NUM_WORKERS = 2

LABEL_SMOOTHING = 0.05
RDROP_ALPHA = 0.5
DROPOUT = 0.15

RESULT_DIR = Path("experiment_results")
RESULT_DIR.mkdir(parents=True, exist_ok=True)

def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
print("device:", device)
print("GPU:", gpu_name)
print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("accelerate:", accelerate.__version__)

## 6. LabelEncoder 생성 및 train에 없는 class 사전 검사

In [ ]:
TASK_COLUMNS = {
    "intent": "label",
    "menu": "menu",
    "temperature": "temperature",
    "quantity": "quantity",
}

train_order_mask = train_df["label"].eq("ORDER")
if train_order_mask.sum() == 0:
    raise ValueError("train.csv에 ORDER 행이 없습니다.")

label_encoders = {}

intent_le = LabelEncoder()
intent_le.fit(train_df["label"].astype(str))
label_encoders["intent"] = intent_le

# slot label space는 실제 학습 대상인 ORDER 행만으로 정의한다.
for task in ["menu", "temperature", "quantity"]:
    col = TASK_COLUMNS[task]
    le = LabelEncoder()
    le.fit(train_df.loc[train_order_mask, col].astype(str))
    label_encoders[task] = le

def assert_no_unseen_classes(split_name: str, df: pd.DataFrame):
    problems = {}

    known_intents = set(label_encoders["intent"].classes_)
    unseen_intents = sorted(set(df["label"].astype(str)) - known_intents)
    if unseen_intents:
        problems["intent"] = unseen_intents

    order_mask = df["label"].eq("ORDER")
    for task in ["menu", "temperature", "quantity"]:
        col = TASK_COLUMNS[task]
        known = set(label_encoders[task].classes_)
        unseen = sorted(set(df.loc[order_mask, col].astype(str)) - known)
        if unseen:
            problems[task] = unseen

    if problems:
        detail = "\n".join(f"- {task}: {classes}" for task, classes in problems.items())
        raise ValueError(
            f"{split_name}.csv에 train.csv에 없던 class가 있습니다.\n{detail}"
        )

assert_no_unseen_classes("valid", valid_df)
assert_no_unseen_classes("test", test_df)

ORDER_ID = int(label_encoders["intent"].transform(["ORDER"])[0])

label_maps = {
    task: {
        "classes": le.classes_.tolist(),
        "class_to_id": {str(c): int(i) for i, c in enumerate(le.classes_)},
        "id_to_class": {str(i): str(c) for i, c in enumerate(le.classes_)},
    }
    for task, le in label_encoders.items()
}

with open(RESULT_DIR / "label_maps.json", "w", encoding="utf-8") as f:
    json.dump(label_maps, f, ensure_ascii=False, indent=2)

for task, le in label_encoders.items():
    print(f"{task}: {len(le.classes_)} classes -> {le.classes_.tolist()}")

## 7. Tokenizer, PyTorch Dataset, DataLoader

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(ENCODER_NAME)
tokenizer.save_pretrained(RESULT_DIR / "tokenizer")

class NLUDataset(Dataset):
    def __init__(self, df: pd.DataFrame, tokenizer, max_length: int):
        self.df = df.reset_index(drop=True)
        self.encodings = tokenizer(
            self.df["text"].astype(str).tolist(),
            truncation=True,
            padding="max_length",
            max_length=max_length,
            return_tensors="pt",
        )

        self.intent_ids = torch.tensor(
            label_encoders["intent"].transform(self.df["label"].astype(str)),
            dtype=torch.long,
        )
        self.is_order = self.df["label"].eq("ORDER").to_numpy()

        # 비-ORDER slot 값은 loss에서 사용되지 않으므로 안전한 0 placeholder를 넣는다.
        self.slot_ids = {}
        for task in ["menu", "temperature", "quantity"]:
            col = TASK_COLUMNS[task]
            ids = np.zeros(len(self.df), dtype=np.int64)
            if self.is_order.any():
                ids[self.is_order] = label_encoders[task].transform(
                    self.df.loc[self.is_order, col].astype(str)
                )
            self.slot_ids[task] = torch.tensor(ids, dtype=torch.long)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item.update({
            "intent_labels": self.intent_ids[idx],
            "menu_labels": self.slot_ids["menu"][idx],
            "temperature_labels": self.slot_ids["temperature"][idx],
            "quantity_labels": self.slot_ids["quantity"][idx],
            "order_mask": torch.tensor(self.is_order[idx], dtype=torch.bool),
            "row_id": torch.tensor(idx, dtype=torch.long),
        })
        return item

train_dataset = NLUDataset(train_df, tokenizer, MAX_LENGTH)
valid_dataset = NLUDataset(valid_df, tokenizer, MAX_LENGTH)
test_dataset  = NLUDataset(test_df, tokenizer, MAX_LENGTH)

generator = torch.Generator()
generator.manual_seed(SEED)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    generator=generator,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)
valid_loader = DataLoader(
    valid_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

print("train batches:", len(train_loader))
print("valid batches:", len(valid_loader))
print("test batches :", len(test_loader))

## 8. 모델 클래스 정의

### 핵심 구조
1. **Shared encoder**: 한 번의 koELECTRA 추론 결과를 네 task가 공유합니다.
2. **Task-specific attention pooling**: intent/menu/temperature/quantity가 각자 중요한 토큰을 선택합니다.
3. **Gated CLS fusion**: `[CLS]` 표현과 attention pooled 표현을 학습 가능한 gate로 결합합니다.
4. **Intent-conditioned slots**: intent 확률 분포를 hidden vector로 투영해 slot 표현에 더합니다.
5. **ORDER-only slot loss**: 정답 intent가 ORDER인 행에서만 세 slot loss를 계산합니다.

In [ ]:
class GatedAttentionPooler(nn.Module):
    def __init__(self, hidden_size: int, dropout: float):
        super().__init__()
        self.attn = nn.Linear(hidden_size, 1)
        self.gate = nn.Linear(hidden_size * 2, hidden_size)
        self.norm = nn.LayerNorm(hidden_size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, hidden_states, attention_mask):
        # padding token에는 매우 작은 점수를 부여해 attention에서 제외한다.
        scores = self.attn(hidden_states).squeeze(-1)
        scores = scores.masked_fill(attention_mask == 0, -1e4)
        weights = torch.softmax(scores, dim=-1)
        attended = torch.bmm(weights.unsqueeze(1), hidden_states).squeeze(1)

        cls = hidden_states[:, 0]
        gate = torch.sigmoid(self.gate(torch.cat([cls, attended], dim=-1)))
        fused = gate * attended + (1.0 - gate) * cls
        return self.dropout(self.norm(fused))

class HierarchicalGatedMultiTaskModel(nn.Module):
    def __init__(self, encoder_name: str, num_labels: dict, dropout: float = 0.15):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(encoder_name)
        hidden = self.encoder.config.hidden_size

        self.poolers = nn.ModuleDict({
            task: GatedAttentionPooler(hidden, dropout)
            for task in ["intent", "menu", "temperature", "quantity"]
        })

        self.intent_head = nn.Sequential(
            nn.Linear(hidden, hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, num_labels["intent"]),
        )

        # intent 확률 전체를 연속적인 context vector로 변환한다.
        self.intent_context = nn.Sequential(
            nn.Linear(num_labels["intent"], hidden),
            nn.Tanh(),
        )

        self.slot_heads = nn.ModuleDict({
            task: nn.Sequential(
                nn.Linear(hidden, hidden),
                nn.GELU(),
                nn.Dropout(dropout),
                nn.Linear(hidden, num_labels[task]),
            )
            for task in ["menu", "temperature", "quantity"]
        })

    def forward(self, input_ids, attention_mask, token_type_ids=None):
        encoder_kwargs = {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
        }
        if token_type_ids is not None:
            encoder_kwargs["token_type_ids"] = token_type_ids

        hidden_states = self.encoder(**encoder_kwargs).last_hidden_state

        intent_repr = self.poolers["intent"](hidden_states, attention_mask)
        intent_logits = self.intent_head(intent_repr)

        # soft distribution을 사용해 train/inference 사이의 hard-label 불일치를 피한다.
        intent_probs = torch.softmax(intent_logits, dim=-1)
        intent_ctx = self.intent_context(intent_probs)

        logits = {"intent": intent_logits}
        for task in ["menu", "temperature", "quantity"]:
            slot_repr = self.poolers[task](hidden_states, attention_mask)
            logits[task] = self.slot_heads[task](slot_repr + intent_ctx)

        return logits

num_labels = {task: len(le.classes_) for task, le in label_encoders.items()}
model = HierarchicalGatedMultiTaskModel(
    ENCODER_NAME,
    num_labels=num_labels,
    dropout=DROPOUT,
).to(device)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"trainable parameters: {trainable_params:,}")

## 9. Loss와 R-Drop 함수

- supervised loss = intent CE + ORDER 행의 menu/temperature/quantity CE 평균
- 각 CE에는 작은 label smoothing을 적용합니다.
- R-Drop은 같은 batch를 dropout 상태에서 두 번 통과시켜 두 예측 분포가 지나치게 달라지지 않도록 symmetric KL을 추가합니다.
- slot KL 역시 ORDER 행에서만 계산합니다.

In [ ]:
def supervised_loss(logits, batch):
    intent_loss = F.cross_entropy(
        logits["intent"],
        batch["intent_labels"],
        label_smoothing=LABEL_SMOOTHING,
    )

    order_mask = batch["order_mask"]
    slot_losses = []
    if order_mask.any():
        for task in ["menu", "temperature", "quantity"]:
            slot_losses.append(
                F.cross_entropy(
                    logits[task][order_mask],
                    batch[f"{task}_labels"][order_mask],
                    label_smoothing=LABEL_SMOOTHING,
                )
            )

    slot_loss = torch.stack(slot_losses).mean() if slot_losses else intent_loss.new_zeros(())
    total = intent_loss + slot_loss
    return total, {
        "intent_loss": float(intent_loss.detach().cpu()),
        "slot_loss": float(slot_loss.detach().cpu()),
    }

def symmetric_kl(logits_a, logits_b):
    log_p = F.log_softmax(logits_a, dim=-1)
    log_q = F.log_softmax(logits_b, dim=-1)
    p = log_p.exp()
    q = log_q.exp()
    kl_pq = F.kl_div(log_p, q, reduction="batchmean")
    kl_qp = F.kl_div(log_q, p, reduction="batchmean")
    return 0.5 * (kl_pq + kl_qp)

def rdrop_loss(logits_a, logits_b, order_mask):
    losses = [symmetric_kl(logits_a["intent"], logits_b["intent"])]
    if order_mask.any():
        for task in ["menu", "temperature", "quantity"]:
            losses.append(
                symmetric_kl(
                    logits_a[task][order_mask],
                    logits_b[task][order_mask],
                )
            )
    return torch.stack(losses).mean()

## 10. 공통 평가 지표 함수

In [ ]:
def safe_macro_f1(y_true, y_pred):
    return float(f1_score(y_true, y_pred, average="macro", zero_division=0))

def compute_metrics_from_predictions(df, pred_ids):
    true_intent = label_encoders["intent"].transform(df["label"].astype(str))
    pred_intent = np.asarray(pred_ids["intent"])
    order_mask = df["label"].eq("ORDER").to_numpy()

    metrics = {
        "intent_accuracy": float(accuracy_score(true_intent, pred_intent)),
        "intent_macro_f1": safe_macro_f1(true_intent, pred_intent),
    }

    decoded = {
        "true_intent": label_encoders["intent"].inverse_transform(true_intent),
        "pred_intent": label_encoders["intent"].inverse_transform(pred_intent),
    }

    slot_correct = {}
    for task in ["menu", "temperature", "quantity"]:
        col = TASK_COLUMNS[task]
        true_slot = label_encoders[task].transform(
            df.loc[order_mask, col].astype(str)
        )
        pred_slot_all = np.asarray(pred_ids[task])
        pred_slot = pred_slot_all[order_mask]

        metrics[f"{task}_accuracy"] = float(accuracy_score(true_slot, pred_slot))
        metrics[f"{task}_macro_f1"] = safe_macro_f1(true_slot, pred_slot)

        decoded[f"true_{task}"] = np.full(len(df), "NONE", dtype=object)
        decoded[f"pred_{task}"] = np.full(len(df), "NONE", dtype=object)
        decoded[f"true_{task}"][order_mask] = label_encoders[task].inverse_transform(true_slot)
        decoded[f"pred_{task}"][order_mask] = label_encoders[task].inverse_transform(pred_slot)

        correct = np.ones(len(df), dtype=bool)
        correct[order_mask] = true_slot == pred_slot
        slot_correct[task] = correct

    intent_correct = true_intent == pred_intent
    all_slots_correct = (
        slot_correct["menu"]
        & slot_correct["temperature"]
        & slot_correct["quantity"]
    )

    order_exact = all_slots_correct[order_mask]
    frame_correct = np.where(
        order_mask,
        intent_correct & all_slots_correct,
        intent_correct,
    )

    metrics["order_exact_match"] = float(order_exact.mean()) if order_mask.any() else 0.0
    metrics["frame_accuracy"] = float(frame_correct.mean())

    return metrics, decoded, {
        "intent_correct": intent_correct,
        "menu_correct": slot_correct["menu"],
        "temperature_correct": slot_correct["temperature"],
        "quantity_correct": slot_correct["quantity"],
        "order_exact_match": np.where(order_mask, all_slots_correct, False),
        "frame_correct": frame_correct,
    }

@torch.no_grad()
def predict_loader(model, loader):
    model.eval()
    collected = {task: [] for task in ["intent", "menu", "temperature", "quantity"]}
    row_ids = []

    for batch in tqdm(loader, leave=False):
        row_ids.extend(batch["row_id"].numpy().tolist())

        inputs = {
            "input_ids": batch["input_ids"].to(device, non_blocking=True),
            "attention_mask": batch["attention_mask"].to(device, non_blocking=True),
        }
        if "token_type_ids" in batch:
            inputs["token_type_ids"] = batch["token_type_ids"].to(device, non_blocking=True)

        logits = model(**inputs)
        for task in collected:
            collected[task].extend(logits[task].argmax(dim=-1).cpu().numpy().tolist())

    order = np.argsort(row_ids)
    return {
        task: np.asarray(values)[order]
        for task, values in collected.items()
    }

def evaluate(model, loader, df):
    pred_ids = predict_loader(model, loader)
    metrics, _, _ = compute_metrics_from_predictions(df, pred_ids)
    return metrics, pred_ids

## 11. Optimizer, scheduler, mixed precision 설정

In [ ]:
from torch.optim import AdamW

optimizer = AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

total_steps = len(train_loader) * EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps,
)

use_amp = torch.cuda.is_available()
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

print("total_steps:", total_steps)
print("warmup_steps:", warmup_steps)
print("mixed precision:", use_amp)

## 12. 학습 함수

In [ ]:
def move_batch_to_device(batch):
    moved = {}
    for k, v in batch.items():
        if k == "row_id":
            moved[k] = v
        else:
            moved[k] = v.to(device, non_blocking=True)
    return moved

def train_one_epoch(model, loader, optimizer, scheduler, scaler):
    model.train()
    running_loss = 0.0
    running_sup = 0.0
    running_kl = 0.0

    progress = tqdm(loader, desc="train", leave=False)
    for batch in progress:
        batch = move_batch_to_device(batch)
        optimizer.zero_grad(set_to_none=True)

        inputs = {
            "input_ids": batch["input_ids"],
            "attention_mask": batch["attention_mask"],
        }
        if "token_type_ids" in batch:
            inputs["token_type_ids"] = batch["token_type_ids"]

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=use_amp,
        ):
            # R-Drop: 동일 입력을 dropout 상태에서 두 번 forward한다.
            logits_a = model(**inputs)
            logits_b = model(**inputs)

            sup_a, _ = supervised_loss(logits_a, batch)
            sup_b, _ = supervised_loss(logits_b, batch)
            sup = 0.5 * (sup_a + sup_b)

            kl = rdrop_loss(logits_a, logits_b, batch["order_mask"])
            loss = sup + RDROP_ALPHA * kl

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        running_loss += float(loss.detach().cpu())
        running_sup += float(sup.detach().cpu())
        running_kl += float(kl.detach().cpu())
        progress.set_postfix(loss=f"{running_loss / (progress.n + 1):.4f}")

    n = max(len(loader), 1)
    return {
        "train_loss": running_loss / n,
        "train_supervised_loss": running_sup / n,
        "train_rdrop_kl": running_kl / n,
        "learning_rate": scheduler.get_last_lr()[0],
    }

## 13. validation frame_accuracy 기준 best model 저장

In [ ]:
BEST_MODEL_PATH = RESULT_DIR / "best_model.pt"
history = []
best_epoch = 0
best_frame = -1.0
best_intent_f1 = -1.0

for epoch in range(1, EPOCHS + 1):
    print(f"\n===== Epoch {epoch}/{EPOCHS} =====")
    train_stats = train_one_epoch(
        model, train_loader, optimizer, scheduler, scaler
    )
    valid_metrics, _ = evaluate(model, valid_loader, valid_df)

    row = {
        "epoch": epoch,
        **train_stats,
        **{f"val_{k}": v for k, v in valid_metrics.items()},
    }
    history.append(row)
    print(json.dumps(row, ensure_ascii=False, indent=2))

    current_frame = valid_metrics["frame_accuracy"]
    current_intent_f1 = valid_metrics["intent_macro_f1"]

    is_better = (
        current_frame > best_frame
        or (
            math.isclose(current_frame, best_frame, rel_tol=0.0, abs_tol=1e-12)
            and current_intent_f1 > best_intent_f1
        )
    )

    if is_better:
        best_epoch = epoch
        best_frame = current_frame
        best_intent_f1 = current_intent_f1

        torch.save({
            "model_state_dict": model.state_dict(),
            "best_epoch": best_epoch,
            "best_validation_frame_accuracy": best_frame,
            "best_validation_intent_macro_f1": best_intent_f1,
            "encoder_name": ENCODER_NAME,
            "num_labels": num_labels,
            "dropout": DROPOUT,
        }, BEST_MODEL_PATH)
        print(f"best model saved -> epoch {best_epoch}")

validation_history_df = pd.DataFrame(history)
validation_history_df.to_csv(
    RESULT_DIR / "validation_history.csv",
    index=False,
    encoding="utf-8-sig",
)

print("best_epoch:", best_epoch)
print("best_validation_frame_accuracy:", best_frame)
print("best_validation_intent_macro_f1:", best_intent_f1)

## 14. best model 다시 불러오기

In [ ]:
del model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

checkpoint = torch.load(BEST_MODEL_PATH, map_location=device)

model = HierarchicalGatedMultiTaskModel(
    checkpoint["encoder_name"],
    num_labels=checkpoint["num_labels"],
    dropout=checkpoint["dropout"],
).to(device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

best_epoch = int(checkpoint["best_epoch"])
best_frame = float(checkpoint["best_validation_frame_accuracy"])

print("best model loaded")
print("best_epoch:", best_epoch)
print("best_validation_frame_accuracy:", best_frame)

## 15. test.csv 최종 평가 — 이 셀을 설정 변경에 사용하지 마세요

아래 셀은 모든 학습과 validation 기반 모델 선택이 끝난 뒤 **최종 평가용으로 한 번 실행**합니다.

In [ ]:
test_pred_ids = predict_loader(model, test_loader)
test_metrics_core, decoded, correctness = compute_metrics_from_predictions(
    test_df, test_pred_ids
)

test_metrics_core

## 16. test 행별 예측 결과 및 오답 분석 CSV 생성

In [ ]:
test_predictions = pd.DataFrame({
    "text": test_df["text"].astype(str),
    "true_intent": decoded["true_intent"],
    "pred_intent": decoded["pred_intent"],
    "true_menu": decoded["true_menu"],
    "pred_menu": decoded["pred_menu"],
    "true_temperature": decoded["true_temperature"],
    "pred_temperature": decoded["pred_temperature"],
    "true_quantity": decoded["true_quantity"],
    "pred_quantity": decoded["pred_quantity"],
    "intent_correct": correctness["intent_correct"],
    "menu_correct": correctness["menu_correct"],
    "temperature_correct": correctness["temperature_correct"],
    "quantity_correct": correctness["quantity_correct"],
    "order_exact_match": correctness["order_exact_match"],
    "frame_correct": correctness["frame_correct"],
})

test_predictions.to_csv(
    RESULT_DIR / "test_predictions.csv",
    index=False,
    encoding="utf-8-sig",
)

test_errors = test_predictions.loc[
    ~test_predictions["frame_correct"]
].copy()
test_errors.to_csv(
    RESULT_DIR / "test_errors.csv",
    index=False,
    encoding="utf-8-sig",
)

print("test predictions:", len(test_predictions))
print("test errors:", len(test_errors))
display(test_predictions.head())

## 17. class_metrics.csv 생성

In [ ]:
class_metric_rows = []

def append_class_metrics(task, y_true, y_pred, class_names):
    precision, recall, f1, support = precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=np.arange(len(class_names)),
        zero_division=0,
    )
    for i, class_name in enumerate(class_names):
        class_metric_rows.append({
            "task": task,
            "class": str(class_name),
            "support": int(support[i]),
            "precision": float(precision[i]),
            "recall": float(recall[i]),
            "f1": float(f1[i]),
        })

true_intent_ids = label_encoders["intent"].transform(test_df["label"].astype(str))
append_class_metrics(
    "intent",
    true_intent_ids,
    test_pred_ids["intent"],
    label_encoders["intent"].classes_,
)

test_order_mask = test_df["label"].eq("ORDER").to_numpy()
for task in ["menu", "temperature", "quantity"]:
    true_ids = label_encoders[task].transform(
        test_df.loc[test_order_mask, TASK_COLUMNS[task]].astype(str)
    )
    pred_ids = test_pred_ids[task][test_order_mask]
    append_class_metrics(
        task,
        true_ids,
        pred_ids,
        label_encoders[task].classes_,
    )

class_metrics_df = pd.DataFrame(class_metric_rows)
class_metrics_df.to_csv(
    RESULT_DIR / "class_metrics.csv",
    index=False,
    encoding="utf-8-sig",
)

display(class_metrics_df.head(20))

## 18. 실험 설정과 결과 JSON 저장

In [ ]:
def get_model_size_mb(path: Path):
    return float(path.stat().st_size / (1024 ** 2))

experiment_config = {
    "experiment_name": EXPERIMENT_NAME,
    "model_architecture": MODEL_ARCHITECTURE,
    "encoder_name": ENCODER_NAME,
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "max_length": MAX_LENGTH,
    "seed": SEED,
    "warmup_ratio": WARMUP_RATIO,
    "gradient_clip_norm": GRAD_CLIP_NORM,
    "label_smoothing": LABEL_SMOOTHING,
    "rdrop_alpha": RDROP_ALPHA,
    "dropout": DROPOUT,
    "train_rows": int(len(train_df)),
    "valid_rows": int(len(valid_df)),
    "test_rows": int(len(test_df)),
    "intent_classes": label_encoders["intent"].classes_.tolist(),
    "menu_classes": label_encoders["menu"].classes_.tolist(),
    "temperature_classes": label_encoders["temperature"].classes_.tolist(),
    "quantity_classes": label_encoders["quantity"].classes_.tolist(),
    "torch_version": torch.__version__,
    "transformers_version": transformers.__version__,
    "accelerate_version": accelerate.__version__,
    "gpu_name": gpu_name,
}

with open(RESULT_DIR / "experiment_config.json", "w", encoding="utf-8") as f:
    json.dump(experiment_config, f, ensure_ascii=False, indent=2)

test_metrics = {
    "experiment_name": EXPERIMENT_NAME,
    "model_architecture": MODEL_ARCHITECTURE,
    "encoder_name": ENCODER_NAME,
    "seed": SEED,
    "best_epoch": best_epoch,
    "best_validation_frame_accuracy": best_frame,
    "intent_accuracy": test_metrics_core["intent_accuracy"],
    "intent_macro_f1": test_metrics_core["intent_macro_f1"],
    "menu_accuracy": test_metrics_core["menu_accuracy"],
    "menu_macro_f1": test_metrics_core["menu_macro_f1"],
    "temperature_accuracy": test_metrics_core["temperature_accuracy"],
    "temperature_macro_f1": test_metrics_core["temperature_macro_f1"],
    "quantity_accuracy": test_metrics_core["quantity_accuracy"],
    "quantity_macro_f1": test_metrics_core["quantity_macro_f1"],
    "order_exact_match": test_metrics_core["order_exact_match"],
    "frame_accuracy": test_metrics_core["frame_accuracy"],
    "model_size_mb": get_model_size_mb(BEST_MODEL_PATH),
}

# JSON과 CSV가 동일한 예측에서 계산됐는지 최종 교차 확인한다.
assert math.isclose(
    test_metrics["frame_accuracy"],
    float(test_predictions["frame_correct"].mean()),
    rel_tol=0.0,
    abs_tol=1e-12,
)
assert math.isclose(
    test_metrics["intent_accuracy"],
    float(test_predictions["intent_correct"].mean()),
    rel_tol=0.0,
    abs_tol=1e-12,
)
order_rows = test_df["label"].eq("ORDER").to_numpy()
assert math.isclose(
    test_metrics["order_exact_match"],
    float(test_predictions.loc[order_rows, "order_exact_match"].mean()),
    rel_tol=0.0,
    abs_tol=1e-12,
)

with open(RESULT_DIR / "test_metrics.json", "w", encoding="utf-8") as f:
    json.dump(test_metrics, f, ensure_ascii=False, indent=2)

print(json.dumps(test_metrics, ensure_ascii=False, indent=2))

## 19. 결과 폴더 검사 및 ZIP 압축

In [ ]:
expected_paths = [
    RESULT_DIR / "best_model.pt",
    RESULT_DIR / "tokenizer",
    RESULT_DIR / "experiment_config.json",
    RESULT_DIR / "label_maps.json",
    RESULT_DIR / "validation_history.csv",
    RESULT_DIR / "test_metrics.json",
    RESULT_DIR / "test_predictions.csv",
    RESULT_DIR / "test_errors.csv",
    RESULT_DIR / "class_metrics.csv",
]

missing_outputs = [str(p) for p in expected_paths if not p.exists()]
if missing_outputs:
    raise FileNotFoundError(f"생성되지 않은 결과 파일/폴더: {missing_outputs}")

zip_path = Path("experiment_results.zip")
if zip_path.exists():
    zip_path.unlink()

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for path in RESULT_DIR.rglob("*"):
        if path.is_file():
            zf.write(path, arcname=path.as_posix())

print("ZIP 생성 완료:", zip_path)
print("\n결과 구조:")
for path in sorted(RESULT_DIR.rglob("*")):
    print("-", path)

## 20. Colab에서 컴퓨터로 ZIP 다운로드

In [ ]:
from google.colab import files
files.download("experiment_results.zip")